# MeChess 9/9: train the clock adapters and let Maia-3 play (GPU notebook)

An adapter around `chessme humanlike-adapter` and `chessme humanlike-selfplay`. It trains the clock-aware adapters (a likelihood-only one and the behaviour-matched one, λ = 0.5) on a frozen Maia-3 23M from notebook 6/7/8's outputs, then lets three players play games from the positions, clocks and ratings of real test games (games the adapters never saw): the frozen model against itself, the likelihood adapter against itself, and the behaviour-matched adapter against itself. Moves are sampled from each model's probabilities; thinking time is drawn from what humans spent in the same situation.

**Before you run:** add as inputs the outputs of notebook 6 (test sets), notebook 7 (scores) and notebook 8 run for the 23M model (the judged samples with 16 candidates); *Accelerator -> GPU*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**. Maia-3 (code and weights) is an external download (AGPL-3.0) and is not redistributed.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
GAMES = 2000                                            # games per match-up
LAMBDA = "0.5"                                          # weight of the behaviour term
MAIA3_COMMIT = ""                                       # empty: the current default branch; a commit hash pins the baseline exactly
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
INPUT_DIR = ""                                          # the output folders of notebooks 6, 7 and 8; empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
def find(name):
    return next(iter(sorted(glob.glob(root + "/**/" + name, recursive=True))), None)
TB, SCORES = find("testset_TB.jsonl.gz"), find("scores_TB_23m.npz")
JUDGED_TRAIN, JUDGED_TEST = find("judged_trainval.npz"), find("judged_test.npz")      # notebook 8 run with MODEL = "23m"
print(TB, SCORES, JUDGED_TRAIN, JUDGED_TEST, sep="\n")
if not all((TB, SCORES, JUDGED_TRAIN, JUDGED_TEST)):
    raise SystemExit("Add the outputs of notebooks 6, 7 and 8 (23M run) as inputs.")
os.makedirs(OUT, exist_ok=True)

## Step 1. Maia-3 (external, not part of this repository)

In [ ]:
sh("git", "clone", "--depth", "1", "https://github.com/CSSLab/maia3.git", "/tmp/maia3")
if MAIA3_COMMIT:
    sh("git", "-C", "/tmp/maia3", "fetch", "--depth", "1", "origin", MAIA3_COMMIT)
    sh("git", "-C", "/tmp/maia3", "checkout", MAIA3_COMMIT)
sh("git", "-C", "/tmp/maia3", "log", "-1", "--format=maia3 commit %h %ad")
os.makedirs("/tmp/weights", exist_ok=True)
sh("curl", "-L", "-s", "-o", "/tmp/weights/maia3-23m.pt", "https://huggingface.co/UofTCSSLab/Maia3-23M/resolve/main/maia3-23m.pt")
sh("sha256sum", "/tmp/weights/maia3-23m.pt")

## Step 2. Train the adapters (they are saved to `adapters/`)

In [ ]:
sh(*CLI, "humanlike-adapter", "--testset", TB, "--scores", SCORES, "--judged", JUDGED_TEST, "--judged-train", JUDGED_TRAIN, "--lambdas", LAMBDA, "--epochs", "15",
   "--boot", "100", "--save-dir", f"{OUT}/adapters", "--out", f"{OUT}/adapter_report.md")

## Step 3. Games: frozen vs frozen, likelihood adapter vs itself, behaviour-matched adapter vs itself

In [ ]:
COMMON = ["--testset", TB, "--maia3-repo", "/tmp/maia3", "--weights-dir", "/tmp/weights", "--device", "cuda", "--games", str(GAMES), "--split", "test", "--seed", "1"]
MODEL = "maia3=23m+history"
LIKELIHOOD = f"{OUT}/adapters/adapter_all_features_temperature_tail_slot_biases.pt"
BEHAVIOUR = f"{OUT}/adapters/behaviour_matched_adapter_lambda_{LAMBDA}.pt"
sh(*CLI, "humanlike-selfplay", *COMMON, "--white", MODEL, "--out", f"{OUT}/games_frozen.jsonl")
sh(*CLI, "humanlike-selfplay", *COMMON, "--white", MODEL, "--white-adapter", LIKELIHOOD, "--black", MODEL, "--black-adapter", LIKELIHOOD, "--out", f"{OUT}/games_likelihood_adapter.jsonl")
sh(*CLI, "humanlike-selfplay", *COMMON, "--white", MODEL, "--white-adapter", BEHAVIOUR, "--black", MODEL, "--black-adapter", BEHAVIOUR, "--out", f"{OUT}/games_behaviour_adapter.jsonl")